# DC-Ops: Train YOLOv8n-seg on Merged Roboflow Datasets

Downloads 4 Roboflow datasets directly, merges into 16 DC-Ops classes, trains YOLOv8n-seg.

**2,036 human-labeled images** from:
- Server Vision (1,293 images, 30 classes)
- PC Ports (327 images, 4 classes)
- Ports ym (138 images, 7 classes)
- Server Detection Fujitsu (54 images, 9 classes)

**Runtime → T4 GPU**

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
# Download all 4 datasets from Roboflow
from roboflow import Roboflow

API_KEY = input('Paste your Roboflow API key: ')
rf = Roboflow(api_key=API_KEY)

print('1/4 Downloading Ports (ym)...')
rf.workspace('ym-pffnw').project('ports-yutnj').version(1).download('yolov8', location='roboflow_ports_ym')

print('2/4 Downloading Server Detection (Fujitsu)...')
rf.workspace('fujitsu-ih8ei').project('server-detection').version(1).download('yolov8', location='roboflow_fujitsu')

print('3/4 Downloading Server Vision (hunters)...')
rf.workspace('hunters-workspace-kqclz').project('server-vision').version(5).download('yolov8', location='roboflow_server_vision')

print('4/4 Downloading PC Ports...')
rf.workspace('ports-gmmmp').project('pc-ports').version(10).download('yolov8', location='roboflow_pc_ports')

print('All datasets downloaded!')

In [ ]:
# Merge datasets with class mapping
import os, shutil, yaml, glob
from pathlib import Path

DC_CLASSES = [
    'server rack', 'compute tray', 'NVLink switch tray', 'network switch',
    'power shelf', 'cable', 'network port', 'LED indicator',
    'label', 'fan', 'cooling manifold', 'cable cartridge',
    'power connector', 'drive bay', 'management port', 'DPU',
]

PORTS_YM_MAP = {0:6, 1:6, 2:6, 3:12, 4:6, 5:6, 6:6}
FUJITSU_MAP = {0:6, 1:6, 2:6, 3:6, 4:6, 5:15, 6:1, 7:1, 8:6}
SERVER_VISION_MAP = {
    0:1, 1:13, 2:1, 3:1, 4:1, 5:1, 6:13, 7:13, 8:13, 9:9,
    10:13, 11:6, 12:6, 13:10, 14:14, 15:6, 16:14, 17:15, 18:13,
    19:12, 20:12, 21:15, 22:4, 23:15, 24:6, 25:15, 26:14, 27:6, 28:6, 29:6
}
PC_PORTS_MAP = {0:6, 1:6, 2:6, 3:6}

DATASETS = [
    ('ports_ym', 'roboflow_ports_ym', PORTS_YM_MAP),
    ('fujitsu', 'roboflow_fujitsu', FUJITSU_MAP),
    ('server_vision', 'roboflow_server_vision', SERVER_VISION_MAP),
    ('pc_ports', 'roboflow_pc_ports', PC_PORTS_MAP),
]

OUT = Path('merged')
for split in ['train', 'val']:
    (OUT / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT / split / 'labels').mkdir(parents=True, exist_ok=True)

def bbox_to_poly(parts):
    cx, cy, w, h = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
    x1, y1 = cx - w/2, cy - h/2
    x2, y2 = cx + w/2, cy + h/2
    return f'{x1:.6f} {y1:.6f} {x2:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x1:.6f} {y2:.6f}'

def remap_label(src, dst, cmap):
    lines = []
    with open(src) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            nc = cmap.get(int(parts[0]))
            if nc is None: continue
            lines.append(f'{nc} {bbox_to_poly(parts[1:5])}')
    if lines:
        with open(dst, 'w') as f: f.write('\n'.join(lines) + '\n')
        return True
    return False

total_t, total_v = 0, 0
for ds_name, ds_path, cmap in DATASETS:
    ds = Path(ds_path)
    if not ds.exists():
        print(f'{ds_name}: NOT FOUND'); continue
    for ss, sd in [('train','train'), ('valid','val'), ('test','train')]:
        img_dir = ds / ss / 'images'
        lbl_dir = ds / ss / 'labels'
        if not img_dir.exists(): continue
        c = 0
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() not in ('.jpg','.jpeg','.png','.webp'): continue
            lbl = lbl_dir / f'{img.stem}.txt'
            if not lbl.exists(): continue
            pre = f'{ds_name}_{ss}_'
            shutil.copy2(img, OUT / sd / 'images' / f'{pre}{img.name}')
            if remap_label(lbl, OUT / sd / 'labels' / f'{pre}{img.stem}.txt', cmap):
                c += 1
        if sd == 'train': total_t += c
        else: total_v += c
        if c > 0: print(f'  {ds_name}/{ss}: {c} -> {sd}')

cfg = {'path': str(OUT.resolve()), 'train': 'train/images', 'val': 'val/images',
       'names': {i: n for i, n in enumerate(DC_CLASSES)}}
with open(OUT / 'dataset.yaml', 'w') as f: yaml.dump(cfg, f)

print(f'\nMerged: {total_t} train, {total_v} val, {total_t+total_v} total')

In [ ]:
# Train YOLOv8n-seg
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')

results = model.train(
    data='merged/dataset.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    project='runs',
    name='dc_ops_merged',
    exist_ok=True,
    device=0,
    patience=20,
    save=True,
    plots=True,
    verbose=True,
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    degrees=10.0,
    scale=0.5,
    fliplr=0.5,
)

In [ ]:
# Validate
import glob as g
best_path = g.glob('runs/**/best.pt', recursive=True)[-1]
print(f'Best model: {best_path}')

best = YOLO(best_path)
metrics = best.val()

DC_CLASSES = [
    'server rack', 'compute tray', 'NVLink switch tray', 'network switch',
    'power shelf', 'cable', 'network port', 'LED indicator',
    'label', 'fan', 'cooling manifold', 'cable cartridge',
    'power connector', 'drive bay', 'management port', 'DPU',
]

print(f'\nmAP50 (box):  {metrics.box.map50:.3f}')
print(f'mAP50-95 (box): {metrics.box.map:.3f}')
print(f'\nPer-class mAP50 (box):')
for i, name in enumerate(DC_CLASSES):
    if i < len(metrics.box.ap50):
        print(f'  {name:25s} {metrics.box.ap50[i]:.3f}')

In [ ]:
# Preview predictions
import matplotlib.pyplot as plt
import cv2

val_imgs = sorted(g.glob('merged/val/images/*.*'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, val_imgs):
    results = best(img_path, conf=0.3)
    annotated = results[0].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(Path(img_path).name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Download trained model
import shutil, os
shutil.copy2(best_path, 'dc_ops_yolov8n_seg_v3.pt')
print(f'Size: {os.path.getsize("dc_ops_yolov8n_seg_v3.pt") / 1e6:.1f} MB')

from google.colab import files
files.download('dc_ops_yolov8n_seg_v3.pt')